In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text('process_date', datetime.now().strftime('%Y-%m-%d'), 'Processing date')

#Paths and configurarions
bronze_table = 'pokemons.bronze.pokemons_raw'
process_date = dbutils.widgets.get('process_date')

bronze_df = spark.table(bronze_table) \
    .filter(F.col('ingestion_date') == process_date)

# Stream
# bronze_df = spark.readStream.table("pokemons.bronze.pokemon_raw") \
#     .filter(F.col("ingestion_date") == process_date)
print(f'DEBUG: process_date={process_date}')
print(f'DEBUG: Size: {bronze_df.count()} registros')


DEBUG: process_date=2026-03-19
DEBUG: Size: 20 registros


In [0]:
pokemon_schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('height', IntegerType(), True),
    StructField('weight', IntegerType(), True),
    StructField('base_experience', IntegerType(), True),
    StructField('is_default', BooleanType(), True),

    #Types
    StructField('types', ArrayType( StructType([
        StructField('slot', IntegerType(), True),
        StructField('type', StructType([
            StructField('name', StringType(), True),
            StructField('url', StringType(), True)
        ]))
    ])), True),

    #Stats
    StructField('stats', ArrayType( StructType([
        StructField('base_stat', IntegerType(), True),
        StructField('effort', IntegerType(), True),
        StructField('stat', StructType([
            StructField('name', StringType(), True),
            StructField('url', StringType(), True)
        ]))
    ])), True),
])

In [0]:
df_parsed = bronze_df.select(
    F.col('pokemon_id'),
    F.from_json(F.col('raw_json'), pokemon_schema).alias('parsed'),
    F.col('ingestion_date')
)
df_silver_pokemon = df_parsed.select(
        F.col('pokemon_id'),
        F.col('parsed.id').alias('api_id'),
        F.lower(F.col('parsed.name')).alias('name_clean'),
        F.col('parsed.height').alias('height_cm'),
        F.col('parsed.weight').alias('weight_kg'),
        F.col('parsed.base_experience').alias('base_xp'),
        F.col("parsed.is_default").alias("is_default_variant"),
        F.col("ingestion_date")
).filter(col("name_clean").isNotNull())

display(df_silver_pokemon.limit(10))


In [0]:
df_silver_pokemon.write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('ingestion_date') \
    .saveAsTable('pokemons.silver.pokemon')

In [0]:
df_silver_types = df_parsed.select(
    F.col('pokemon_id'),
    F.explode(F.col('parsed.types')).alias('type_info'),
    F.col('ingestion_date')
).select(
    F.col('pokemon_id'),
    F.col('type_info.slot').alias('type_slot'),
    F.lower(F.col('type_info.type.name')).alias('type_name'),
    F.col('ingestion_date')
)

display(df_silver_types.limit(5))




pokemon_id,type_slot,type_name,ingestion_date
1,1,grass,2026-03-19
1,2,poison,2026-03-19
2,1,grass,2026-03-19
2,2,poison,2026-03-19
3,1,grass,2026-03-19


In [0]:
df_silver_types.write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('ingestion_date') \
    .saveAsTable('pokemons.silver.types')

In [0]:
df_silver_stats = df_parsed.select(
    F.col('pokemon_id'),
    F.explode(F.col('parsed.stats')).alias('stat_info'),
    F.col('ingestion_date')
).select(
    F.col('pokemon_id'),
    F.lower(F.col('stat_info.stat.name')).alias('stat_name'),
    F.col('stat_info.base_stat').alias('base_stat_value'),
    F.col('stat_info.effort').alias('effort_value'),
    F.col('ingestion_date')
)

display(df_silver_stats.limit(10))



pokemon_id,stat_name,base_stat_value,effort_value,ingestion_date
1,hp,45,0,2026-03-19
1,attack,49,0,2026-03-19
1,defense,49,0,2026-03-19
1,special-attack,65,1,2026-03-19
1,special-defense,65,0,2026-03-19
1,speed,45,0,2026-03-19
2,hp,60,0,2026-03-19
2,attack,62,0,2026-03-19
2,defense,63,0,2026-03-19
2,special-attack,80,1,2026-03-19


In [0]:
df_silver_stats.write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('ingestion_date') \
    .saveAsTable('pokemons.silver.pokemon_stats')